The KYTC GIS team has developed an API that allows the public to process requests for roadway characteristics based on individual latitude and longitude coordinates. Each incident has a latitude and longitude.  While the KSP record has some roadway characteristics, roadway designations can change and to eliminate human error, each of our incidents are processed through the KYTC API and selected roadway characteristics for further analysis and visualizations.

In [1]:
# Import required modules
import os
import sqlite3
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point
import json
import requests
from furl import furl
from time import perf_counter
import numpy as np

In [2]:
# Set Pandas Options to show the max width of the dataframe when printing
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

After connecting to the database, we build the url request string for the API call.  The return keys selected are only part of the available parameters that are frequently used by KYTC to identify characteristics of each 0.5 mile sections of all of the roadway centerlines maintained by the Cabinet.  

In [3]:
# Define the path for the SQLite database
cwd = os.getcwd()
database_path = os.path.join(cwd, 'data', 'crash_data.db')

# Connect to the SQLite database
conn = sqlite3.connect(database_path)

# Build the request url with parameters
url_base = r"https://kytc-api-v100-lts-qrntk7e3ra-uc.a.run.app/api/"

return_keys = r"Cardinality, "\
              r" County_Name, Direction, Government_Level, "\
              r" Grade_Percent, Lane_Width_Feet, Median_Type, "\
              r" Median_Width_Feet, Road_Name, "\
              r" Route, Route_Type, Route_Unique_Identifier, "\
              r" Milepoint, Speed_Limit_Posted_MPH, Traffic_Last_Count, "\
              r" Truck_Weight_Limit_Class, Type_Operation, Geometry"

# create a list to store the results in
results = list()
record_count = 0  # Initialize the counter

This function is called for each record in the incident table.

In [4]:
# Function to call API for roadway characteristics from lat/lon values of incidents
def snap_points(row):
    global record_count
    record_count += 1 # Increment the counter

    #  Build the request url with parameters
    url = furl(path=rf"{url_base}route/GetRouteInfoByCoordinates",
               query_params={
               "xcoord": f"{row.get('Longitude', '')}",
               "ycoord": f"{row.get('Latitude', '')}",
               "snap_distance": 100,
               "return_multiple": False,
               "return_m": True,
               "return_keys": return_keys,
               "return_format": "json",
               "request_id": row.get('IncidentID', ''),
               "input_epsg": 4326,
               "output_epsg": 4326})

    print(f"Processing record {record_count}: {row.get('Longitude')}, {row.get('Latitude')}")

    # Send the request
    res = requests.get(url.tostr())

    if res.status_code == 200:
        res = json.loads(res.content.decode('utf-8'))
        if 'Route_Info' in res:
            route_info = res['Route_Info']
            route_info['IncidentID'] = int(row.get('IncidentID'))
            results.append(route_info)
        elif 'Info' in res:
            if 'request_id' in res:
                print(res['Info'], int(res['request_id']))
            else:
                print(res['Info'])
    else:
        print(res)

Because we want to use the geometry to create the folium map, we create the geometry stored in the GeoJSON file exported after all the requests have finished.

In [5]:
# Function to parse the 'POINT Z (x y z)' format
def parse_point_z(geometry):
    coords = geometry.replace('POINT Z (', '').replace(')', '').split()
    return Point(float(coords[0]), float(coords[1]), float(coords[2]))

In [6]:
# Main
if __name__ == '__main__':

    # Connect to SQLite database and read data into a pandas dataframe
    query = "SELECT IncidentID, Latitude, Longitude FROM ksp_incidents;"
    df = pd.read_sql_query(query, conn)

    # Count the number of incidents to process
    number_of_rows = len(df)
    print(f"Number of rows to process: {number_of_rows}")

    # Create a Request ID
    df['Request_ID'] = df.index.astype(str)

    # Print the first 5 rows of the dataframe
    print(df)

    # Start the stopwatch / counter
    perf_counter_start = perf_counter()

    # Snap the points by sending requests to the API
    df.apply(snap_points, axis=1)

    # Stop the stopwatch / counter
    perf_counter_stop = perf_counter()

    # Put the results into a pandas dataframe
    df_results = pd.DataFrame(results)


Number of rows to process: 4105
      IncidentID   Latitude  Longitude Request_ID
0       32655798  37.898087 -85.698205          0
1       32660760  38.242908 -85.503807          1
2       32660920  38.218664 -85.506102          2
3       32646189  37.873039 -85.703127          3
4       32654634  38.383366 -85.415734          4
5       32648383  37.940707 -85.688702          5
6       32676402  38.285470 -85.507078          6
7       32645598  38.080318 -84.470289          7
8       32654632  38.339949 -85.512293          8
9       32642670  37.885472 -85.698862          9
10      32647732  38.261724 -85.502306         10
11      32660684  38.222814 -85.512488         11
12      32636060  37.897062 -85.698731         12
13      32731609  37.825593 -85.723168         13
14      32645037  37.874671 -85.702756         14
15      32634337  37.886750 -85.698861         15
16      32639784  38.218659 -85.506126         16
17      32643976  38.243698 -85.503701         17
18      32630843  

In [7]:
    # This is for testing only - you don't necessarily want to!
    # print(df)
    # print(df_results)
    # print(df_results.columns)

In [8]:
    # Remove records where 'Geometry' is null
    df_results = df_results.dropna(subset=['Geometry'])

    # Filter rows where Geometry is null
    null_geometry_rows = df_results[df_results['Geometry'].isnull()]

    # Display the rows with null Geometry
    print(null_geometry_rows)

Empty DataFrame
Columns: [Cardinality, County_Name, Direction, Government_Level, Grade_Percent, Lane_Width_Feet, Median_Type, Median_Width_Feet, Road_Name, Route, Route_Type, Route_Unique_Identifier, Milepoint, Speed_Limit_Posted_MPH, Traffic_Last_Count, Truck_Weight_Limit_Class, Type_Operation, Geometry, IncidentID]
Index: []


In [9]:
    # Convert the 'Geometry' column from WKT to shapely geometries
    df_results['Geometry'] = df_results['Geometry'].apply(wkt.loads)

    # Convert to GeoDataFrame
    gdf = gpd.GeoDataFrame(df_results, geometry='Geometry')

    # Set the coordinate reference system (CRS) if known; otherwise, use EPSG:4326 (WGS 84)
    gdf.set_crs(epsg=4326, inplace=True)

    # Save to GeoJSON
    geojson_path = os.path.join(cwd, 'data', 'api_clean_data','Roadway_Characteristics_API.geojson')
    gdf.to_file(geojson_path, driver='GeoJSON')


In [10]:
    # Convert the geometries to WKT format for storage in SQLite
    df_results['Geometry'] = df_results['Geometry'].apply(lambda geom: geom.wkt)

    # Ensure all columns have types that SQLite supports
    for col in df_results.columns:
        if df_results[col].dtype == object:
            df_results[col] = df_results[col].astype(str)

    # Write the df_results dataframe to the SQLite table
    df_results.to_sql('roadway_characteristics', conn, if_exists='replace', index=False)

    # Establish a connection to the SQLite database
    conn = sqlite3.connect(database_path)

    # Write the dataframe to a table named 'Roadway_Characteristics_API'
    df_results.to_sql('Roadway_Characteristics_API', conn, if_exists='replace', index=False)

    # Close the connection
    conn.close()